# ingest data into kafka

In [ ]:
import pandas as pd
from confluent_kafka import Producer
import time

conf = {
    "bootstrap.servers": "kafka-test:9092",
    "client.id": "data_producer",
    "batch.size": 1048576,          # 1 MB
    "linger.ms": 50,
    "acks": "all",
    "retries": 5,
    "compression.type": "zstd",
    "queue.buffering.max.messages": 500000,
    "queue.buffering.max.kbytes": 2097152,  # 2 GB
}

producer = Producer(conf)

def delivery_report(err, msg):
    if err is not None:
        print("Delivery failed:", err)

csv_file = "E:/UNIVERSITY_SUBJECTS/projects/MMD_10_PTIT_2026/csv_data/test_data/northVN_dataAIR.csv"
chunksize = 50000
line_count = 0
bytes_sent = 0

start_time = time.time()

for chunk in pd.read_csv(csv_file, chunksize=chunksize):
    json_lines = chunk.to_json(orient="records", lines=True).splitlines()
    for message in json_lines:
        message_bytes = message.encode("utf-8")
        # retry nếu queue full
        while True:
            try:
                producer.produce(
                    topic="air_quality_data",
                    value=message_bytes,
                    callback=delivery_report
                )
                break
            except BufferError:
                producer.poll(1)
        line_count += 1
        bytes_sent += len(message_bytes)

        if line_count % 1000 == 0:
            producer.poll(0)

        if line_count % 10000 == 0:
            elapsed = time.time() - start_time
            print(
                f"Sent {line_count:,} records | "
                f"{bytes_sent/1024/1024:.2f} MB | "
                f"{line_count/elapsed:.0f} msg/s | "
                f"Queue: {len(producer)}"
            )

producer.flush()

elapsed = time.time() - start_time

print("\nDONE")
print("Total records:", line_count)
print("Total data:", round(bytes_sent / 1024 / 1024, 2), "MB")
print("Time:", round(elapsed, 2), "seconds")
print("Throughput:", round(line_count / elapsed), "msg/s")

Sent 10,000 records | 4.04 MB | 39557 msg/s | Queue: 10000
Sent 20,000 records | 8.08 MB | 74963 msg/s | Queue: 20000
Sent 30,000 records | 12.13 MB | 106457 msg/s | Queue: 30000
Sent 40,000 records | 16.19 MB | 134950 msg/s | Queue: 40000
Sent 50,000 records | 20.25 MB | 160503 msg/s | Queue: 50000
Sent 60,000 records | 24.32 MB | 100786 msg/s | Queue: 60000
Sent 70,000 records | 28.36 MB | 114505 msg/s | Queue: 70000
Sent 80,000 records | 32.40 MB | 127730 msg/s | Queue: 80000
Sent 90,000 records | 36.45 MB | 140320 msg/s | Queue: 90000
Sent 100,000 records | 40.50 MB | 152344 msg/s | Queue: 100000
Sent 110,000 records | 44.56 MB | 120696 msg/s | Queue: 110000
Sent 120,000 records | 48.62 MB | 129461 msg/s | Queue: 120000
Sent 130,000 records | 52.70 MB | 138088 msg/s | Queue: 130000
Sent 140,000 records | 56.79 MB | 146396 msg/s | Queue: 140000
Sent 150,000 records | 60.88 MB | 144769 msg/s | Queue: 115350
Sent 160,000 records | 64.95 MB | 114514 msg/s | Queue: 4432
Sent 170,000 rec